In [50]:
import torch 
import torch.nn as nn
import torch.optim as optim


In [51]:
import torchvision
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

In [52]:
trainset=MNIST(root="./data",train=True,download=True,transform=transform)
testset=MNIST(root="./data",train=False,download=True,transform=transform)

In [53]:
print(len(trainset))
print(len(testset))

60000
10000


In [54]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(trainset,batch_size=64,shuffle=False)

In [55]:
image, label = trainset[0]

print(type(image))
print(image.size())   
print(label)

<class 'torch.Tensor'>
torch.Size([1, 28, 28])
5


In [56]:
print(trainset.data.shape)
print(trainset.targets.shape)

torch.Size([60000, 28, 28])
torch.Size([60000])


In [57]:
print(trainset.classes)

['0 - zero', '1 - one', '2 - two', '3 - three', '4 - four', '5 - five', '6 - six', '7 - seven', '8 - eight', '9 - nine']


In [58]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.con_lay=nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )
        self.full_con=nn.Sequential(
            nn.Linear(3*3*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        x=self.con_lay(x)
        x=x.view(x.size(0),-1)
        x=self.full_con(x)
        return x      


In [59]:
model=CNN()
criterion=nn.CrossEntropyLoss()
optimizers=optim.Adam(model.parameters())

In [60]:
model.train()
epochs=10
for epoch in range(epochs):
    for x,y in trainloader:
        running_loss=0
        optimizers.zero_grad()
        outputs=model.forward(x)
        loss=criterion(outputs,y)
        loss.backward()
        optimizers.step()
        running_loss+=loss.item()
    epoch_loss=running_loss/len(trainloader)
    print(f"epoch={epoch+1}/{epochs} && training loss={epoch_loss}")

epoch=1/10 && training loss=5.881690870978431e-05
epoch=2/10 && training loss=4.467842325981238e-06
epoch=3/10 && training loss=1.8874705304453241e-06
epoch=4/10 && training loss=2.9241026782277805e-05
epoch=5/10 && training loss=1.4729367922554647e-05
epoch=6/10 && training loss=1.6123303639164357e-05
epoch=7/10 && training loss=8.18548101642882e-08
epoch=8/10 && training loss=7.819059757398629e-08
epoch=9/10 && training loss=7.363396690789062e-06
epoch=10/10 && training loss=5.238506031894822e-08


In [61]:
model.eval()
correct=0
total=0
with torch.no_grad():
    for x,y in testloader:
        outputs=model.forward(x)
        _,predicted=torch.max(outputs,1)

        total+=y.size(0)
        correct += (predicted == y).sum().item()

    print(f"accuracy={correct/total*100}")
        

accuracy=99.83666666666666
